# 04. Modeling, Hyperparameter Tuning, and Multi-Model Validation
**CSE437 Final Project: Disparity and Error Analysis in Linear Income Classification**  
*Texas 2023 ACS 1-Year PUMS Microdata*

---

### Purpose and Methodological Justifications
1. **Trivial Baseline Benchmark:** Evaluates a Zero-R / Majority-Class predictor (predicting Class 0 for all instances) to establish the minimum performance floor. Under a 74/26 class distribution, an uninformative model scores ~74% accuracy but achieves an F1-score of 0.00%, illustrating why minority-class F1 is the mandated primary metric[cite: 1, 9].
2. **Primary Model Family (`LinearSVC`):** Implements a Support Vector Classifier with L2-penalty soft-margin loss[cite: 1, 9]. Primal optimization (`dual=False`) is selected because the training volume ($N = 24,000$) significantly exceeds feature dimensionality ($D = 28$)[cite: 1, 9].
3. **Hyperparameter Grid Search:** Conducts a 3-fold stratified cross-validation over $C \in \{0.01, 0.1, 1.0, 10.0\}$ using F1 scoring[cite: 1, 9]. The logarithmic grid tests the full regularization trajectory from heavy margin penalty to soft-margin stabilization[cite: 1].
4. **Multi-Model Validation (`LogisticRegression`):** Benchmarks against an independent linear model family based on maximum likelihood estimation to confirm whether margin geometry introduces idiosyncratic classification biases[cite: 1, 9].
5. **Model Serialization & Error Tagging:** Serializes trained estimators to `models/` using `joblib` and annotates test instances with functional margin decision scores and discrete error categories (`False_Negative`, `False_Positive`, `Correct`)[cite: 5, 9].

In [ ]:
import os
import sys
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC

# Portable path resolution relative to repository root
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models"

# Ensure target directories exist
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"[STATUS] Project root resolved to: {REPO_ROOT.resolve()}")
print(f"[STATUS] Reading split matrices from: {PROCESSED_DATA_DIR.resolve()}")
print(f"[STATUS] Serialized models directory: {MODELS_DIR.resolve()}")

## 1. Data Ingestion & Partition Verification
We load the preprocessed feature matrices, target vectors, and metadata tables produced in Notebook 03[cite: 8].

In [ ]:
# Ingest preprocessed splits from data/processed/[cite: 8, 9]
X_train = pd.read_csv(PROCESSED_DATA_DIR / "X_train.csv")
X_test = pd.read_csv(PROCESSED_DATA_DIR / "X_test.csv")
y_train = pd.read_csv(PROCESSED_DATA_DIR / "y_train.csv").squeeze("columns")
y_test = pd.read_csv(PROCESSED_DATA_DIR / "y_test.csv").squeeze("columns")
test_meta = pd.read_csv(PROCESSED_DATA_DIR / "test_metadata.csv")

print("=" * 65)
print("SPLIT MATRIX VERIFICATION")
print("=" * 65)
print(f"X_train Shape: {X_train.shape} | y_train Observations: {len(y_train):,}")
print(f"X_test Shape:  {X_test.shape}  | y_test Observations:  {len(y_test):,}")
print(f"Feature Count: {X_train.shape[1]} engineered predictors (Matches Section 4.4)[cite: 1, 8]")
print(f"Train High-Earner Balance: {y_train.mean()*100:.2f}%")
print(f"Test High-Earner Balance:  {y_test.mean()*100:.2f}%")
print("=" * 65)

## 2. Baseline Predictor (Zero-R / Majority-Class Classifier)
We evaluate a trivial majority-class predictor (`DummyClassifier(strategy='most_frequent')`) to define the uninformative baseline[cite: 2].

In [ ]:
# Trivial baseline predicting majority class (Class 0: Standard Earner)
dummy_baseline = DummyClassifier(strategy="most_frequent")
dummy_baseline.fit(X_train, y_train)
y_pred_dummy = dummy_baseline.predict(X_test)

baseline_acc = accuracy_score(y_test, y_pred_dummy)
baseline_prec = precision_score(y_test, y_pred_dummy, zero_division=0)
baseline_rec = recall_score(y_test, y_pred_dummy, zero_division=0)
baseline_f1 = f1_score(y_test, y_pred_dummy, zero_division=0)

print("=" * 65)
print("TRIVIAL MAJORITY-CLASS BASELINE METRICS (Report Section 5.2)[cite: 1, 2]")
print("=" * 65)
print(f"Accuracy:           {baseline_acc * 100:.2f}%")
print(f"Precision (Class 1): {baseline_prec * 100:.2f}%")
print(f"Recall (Class 1):    {baseline_rec * 100:.2f}%")
print(f"F1-Score (Class 1):  {baseline_f1 * 100:.2f}%")
print("=" * 65)

## 3. Hyperparameter Tuning: LinearSVC Grid Search
We execute a 3-fold stratified `GridSearchCV` over the regularization parameter $C \in \{0.01, 0.1, 1.0, 10.0\}$ with `scoring='f1'` and `dual=False`[cite: 1, 9].

In [ ]:
print("=" * 65)
print("GRID SEARCH: LinearSVC (3-Fold CV, Scoring = F1)[cite: 1, 9]")
print("=" * 65)

param_grid = {"C": [0.01, 0.1, 1.0, 10.0]}
base_svm = LinearSVC(dual=False, max_iter=5000, random_state=42)

grid_search = GridSearchCV(
    estimator=base_svm,
    param_grid=param_grid,
    cv=3,
    scoring="f1",
    n_jobs=-1,
    return_train_score=True,
)
grid_search.fit(X_train, y_train)

# Display complete parameter trajectory[cite: 1, 9]
cv_results = pd.DataFrame({
    "C": [p["C"] for p in grid_search.cv_results_["params"]],
    "Mean Validation F1": grid_search.cv_results_["mean_test_score"],
    "Std Validation F1": grid_search.cv_results_["std_test_score"],
    "Mean Train F1": grid_search.cv_results_["mean_train_score"],
})
print(cv_results.to_string(index=False))
print("-" * 65)
print(f"Optimal Parameter:        C = {grid_search.best_params_['C']}[cite: 1, 9]")
print(f"Best Mean Validation F1:  {grid_search.best_score_:.4f} (Matches Report Section 6.3)[cite: 1, 9]")
print("=" * 65)

best_svm = grid_search.best_estimator_

## 4. Multi-Model Validation: Logistic Regression Comparison
We train an L2-regularized `LogisticRegression` model on identical splits to benchmark linear decision boundaries across independent model families[cite: 1, 9].

In [ ]:
# Fit benchmark model family[cite: 1, 9]
log_reg = LogisticRegression(max_iter=5000, random_state=42)
log_reg.fit(X_train, y_train)

# Generate test predictions for both families[cite: 9]
y_pred_svm = best_svm.predict(X_test)
y_pred_lr = log_reg.predict(X_test)
decision_scores_svm = best_svm.decision_function(X_test)

# Build Report Section 7.1 performance table[cite: 1, 9]
model_comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision (Class 1)", "Recall (Class 1)", "F1-Score (Class 1)"],
    "Trivial Baseline": [
        f"{baseline_acc*100:.2f}%",
        f"{baseline_prec*100:.2f}%",
        f"{baseline_rec*100:.2f}%",
        f"{baseline_f1*100:.2f}%",
    ],
    "LogisticRegression": [
        f"{accuracy_score(y_test, y_pred_lr)*100:.2f}%",
        f"{precision_score(y_test, y_pred_lr)*100:.2f}%",
        f"{recall_score(y_test, y_pred_lr)*100:.2f}%",
        f"{f1_score(y_test, y_pred_lr)*100:.2f}%",
    ],
    "LinearSVC (Tuned C=1.0)": [
        f"{accuracy_score(y_test, y_pred_svm)*100:.2f}%",
        f"{precision_score(y_test, y_pred_svm)*100:.2f}%",
        f"{recall_score(y_test, y_pred_svm)*100:.2f}%",
        f"{f1_score(y_test, y_pred_svm)*100:.2f}%",
    ],
})

print("=" * 65)
print("TEST SET PERFORMANCE COMPARISON (Report Section 7.1)[cite: 1, 9]")
print("=" * 65)
print(model_comparison.to_string(index=False))
print("=" * 65)

# Detailed LinearSVC classification report[cite: 1, 9]
print("\nLinearSVC Classification Report (N=6,000):")
print(classification_report(y_test, y_pred_svm, target_names=["Standard Earner (0)", "High Earner (1)"], digits=4))

## 5. Model Serialization and Error Annotation Export
We serialize both trained models to `models/` using `joblib` and save annotated prediction vectors with margin scores for downstream error auditing in Notebook 05[cite: 5, 9].

In [ ]:
# 1. Save trained models to models/ directory[cite: 5]
svm_model_path = MODELS_DIR / "linear_svc_tuned.joblib"
lr_model_path = MODELS_DIR / "logistic_regression_baseline.joblib"

joblib.dump(best_svm, svm_model_path)
joblib.dump(log_reg, lr_model_path)

print(f"[SUCCESS] Serialized LinearSVC model saved to:\n -> {svm_model_path.resolve()}")
print(f"[SUCCESS] Serialized LogisticRegression model saved to:\n -> {lr_model_path.resolve()}")

# 2. Enrich test metadata with predictions, decision scores, and discrete error tags[cite: 9]
test_meta["PRED_HIGH_EARNER"] = y_pred_svm
test_meta["DECISION_SCORE"] = decision_scores_svm
test_meta["CORRECT"] = (test_meta["HIGH_EARNER_ACTUAL"] == y_pred_svm).astype(int)

test_meta["ERROR_TYPE"] = "Correct"
test_meta.loc[
    (test_meta["HIGH_EARNER_ACTUAL"] == 1) & (test_meta["PRED_HIGH_EARNER"] == 0),
    "ERROR_TYPE",
] = "False_Negative"
test_meta.loc[
    (test_meta["HIGH_EARNER_ACTUAL"] == 0) & (test_meta["PRED_HIGH_EARNER"] == 1),
    "ERROR_TYPE",
] = "False_Positive"

# 3. Export evaluated data artifacts[cite: 9]
eval_output_path = PROCESSED_DATA_DIR / "test_predictions_evaluated.csv"
comparison_output_path = PROCESSED_DATA_DIR / "model_family_comparison.csv"

test_meta.to_csv(eval_output_path, index=False)
model_comparison.to_csv(comparison_output_path, index=False)

print("\n" + "=" * 65)
print(f"[SUCCESS] Test predictions and error metadata exported to:\n -> {eval_output_path.resolve()}")
print("=" * 65)

# Error class distribution check
print("Test Error Breakdown:")
print(test_meta["ERROR_TYPE"].value_counts())